In [ ]:
import plotly.graph_objects as go
import numpy as np

# Plot size
plot_size = 600

# Parameters for the circular track
radius_inner = 12  # inner radius of track
radius_outer = 15  # outer radius of track
theta = np.linspace(0, 2 * np.pi, 100)

# Sensor dimensions
sensor_width = 1.5
gap = 1.5

# 4-sensor layout (left to right): A, B, C, D
# B and C are inner sensors (detect center line)
# A and D are outer sensors (detect edge deviation)
# Sensors are symmetric about the track center

track_center = 13.5  # x-position of track center (midpoint of line width)

# Inner sensors B and C — NO gap between them
# They meet exactly at track_center; combined width covers the full 30mm line
Bx2 = track_center                            # B right edge (touches C, no gap)
Bx1 = Bx2 - sensor_width                      # B left edge

Cx1 = track_center                            # C left edge (touches B, no gap)
Cx2 = Cx1 + sensor_width                      # C right edge

# Outer sensors A and D — separated from inner sensors by 'gap'
# RT RT TR TR pattern: gap prevents cross-talk between adjacent sensors
Ax2 = Bx1 - gap                               # A right edge
Ax1 = Ax2 - sensor_width                      # A left edge

Dx1 = Cx2 + gap                               # D left edge
Dx2 = Dx1 + sensor_width                      # D right edge

# Verify combined B+C width equals track width
assert abs((Cx2 - Bx1) - (radius_outer - radius_inner)) < 0.01, \
    f'B+C width {Cx2-Bx1:.2f} does not match track width {radius_outer-radius_inner}'

print(f'Sensor A: {Ax1:.2f} — {Ax2:.2f}  (center: {(Ax1+Ax2)/2:.2f})')
print(f'Sensor B: {Bx1:.2f} — {Bx2:.2f}  (center: {(Bx1+Bx2)/2:.2f})')
print(f'Sensor C: {Cx1:.2f} — {Cx2:.2f}  (center: {(Cx1+Cx2)/2:.2f})')
print(f'Sensor D: {Dx1:.2f} — {Dx2:.2f}  (center: {(Dx1+Dx2)/2:.2f})')

# Circle coordinates
x_inner = radius_inner * np.cos(theta)
y_inner = radius_inner * np.sin(theta)
x_outer = radius_outer * np.cos(theta)
y_outer = radius_outer * np.sin(theta)

# y values to sweep through
y_values = np.arange(0, 12, 0.5)
x_outer_vals = np.sqrt(radius_outer**2 - y_values**2)
x_inner_vals = np.sqrt(radius_inner**2 - y_values**2)

# Helper: sensor is ON (red) if it overlaps the track band [x_inner, x_outer]
def sensor_color(sx1, sx2, x_in, x_out):
    overlaps = sx1 < x_out and sx2 > x_in
    return '#FF0000' if overlaps else '#00CED1'  # red=ON, cyan=OFF

# Build figure
fig = go.Figure()

# Track circles
fig.add_trace(go.Scatter(x=x_inner, y=y_inner, mode='lines',
                         name='Inner radius (12)', line=dict(color='gray', dash='dot')))
fig.add_trace(go.Scatter(x=x_outer, y=y_outer, mode='lines',
                         name='Outer radius (15)', line=dict(color='gray', dash='dot')))

# Animated sensor traces for each y position
for y, x_in, x_out in zip(y_values, x_inner_vals, x_outer_vals):
    distance = x_out - x_in

    # Track band line
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color='black', width=2),
        x=[x_in, x_out], y=[y, y],
        mode='lines+text+markers',
        text=['', f'd = {distance:.2f} mm'],
        textposition='top center',
        name='Track width'
    ))

    # Sensor A
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color=sensor_color(Ax1, Ax2, x_in, x_out), width=5),
        x=[Ax1, Ax2], y=[y, y], mode='lines',
        name='A'
    ))

    # Sensor B
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color=sensor_color(Bx1, Bx2, x_in, x_out), width=5),
        x=[Bx1, Bx2], y=[y, y], mode='lines',
        name='B'
    ))

    # Sensor C
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color=sensor_color(Cx1, Cx2, x_in, x_out), width=5),
        x=[Cx1, Cx2], y=[y, y], mode='lines',
        name='C'
    ))

    # Sensor D
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color=sensor_color(Dx1, Dx2, x_in, x_out), width=5),
        x=[Dx1, Dx2], y=[y, y], mode='lines',
        name='D'
    ))

# Make y=0 frame visible initially (5 traces per frame, starting at index 2)
for i in range(2, 7):
    fig.data[i].visible = True

# Slider
TRACES_PER_FRAME = 5
OFFSET = 2  # first two are the circles

steps = []
for i, y in enumerate(y_values):
    start = OFFSET + i * TRACES_PER_FRAME
    visibility = [False] * len(fig.data)
    visibility[0] = True  # inner circle always visible
    visibility[1] = True  # outer circle always visible
    for j in range(TRACES_PER_FRAME):
        visibility[start + j] = True
    steps.append(dict(
        method='update',
        args=[{'visible': visibility}],
        label=f'{y:.1f}'
    ))

fig.update_layout(
    sliders=[dict(steps=steps, currentvalue={'prefix': 'y = '}, pad={'t': 50})],
    width=plot_size * 2,
    height=plot_size,
    xaxis=dict(range=[0, 25], autorange=False),
    yaxis=dict(range=[0, 16], autorange=False, scaleanchor='x', scaleratio=1),
    title='4-Sensor Layout: Active (Red) vs Inactive (Cyan) along curved track',
    xaxis_title='x position (mm)',
    yaxis_title='y position (mm)',
    legend=dict(orientation='h', y=1.1)
)

fig.show()


In [ ]:
# Print sensor activation table for all y positions
import numpy as np

radius_inner = 12
radius_outer = 15
sensor_width = 1.5
gap = 1.5
track_center = 13.5

# No gap between B and C — they meet at track_center
Bx2 = track_center
Bx1 = Bx2 - sensor_width
Cx1 = track_center
Cx2 = Cx1 + sensor_width
Ax2 = Bx1 - gap
Ax1 = Ax2 - sensor_width
Dx1 = Cx2 + gap
Dx2 = Dx1 + sensor_width

def is_active(sx1, sx2, x_in, x_out):
    return int(sx1 < x_out and sx2 > x_in)

print(f'{'y':>5}  {'x_in':>6}  {'x_out':>6}  {'A':>2}  {'B':>2}  {'C':>2}  {'D':>2}  Pattern')
print('-' * 55)

y_values = np.arange(0, 12, 0.5)
for y in y_values:
    x_in  = np.sqrt(radius_inner**2 - y**2)
    x_out = np.sqrt(radius_outer**2 - y**2)
    A = is_active(Ax1, Ax2, x_in, x_out)
    B = is_active(Bx1, Bx2, x_in, x_out)
    C = is_active(Cx1, Cx2, x_in, x_out)
    D = is_active(Dx1, Dx2, x_in, x_out)
    print(f'{y:>5.1f}  {x_in:>6.2f}  {x_out:>6.2f}  {A:>2}  {B:>2}  {C:>2}  {D:>2}  {A}{B}{C}{D}')
